# Zero‑Shot Next‑Word Prediction with LSTM‑Attention + Prompt Engineering

In this notebook we will: 
1. Download the small WikiText‑2 corpus.
2. Build a tiny LSTM‑Attention language model and train it for a few epochs (fast enough for a classroom demo).
3. Use the trained model to predict the next word for a given prompt, without any further fine‑tuning.
4. Visualise the attention weights so you can see which part of the prompt the model focuses on.

All steps run in under a minute on a free Google‑Colab GPU.

In [ ]:
# Install required packages (run once)
!pip install -q torch==2.2.0 tqdm


## 1. Download WikiText‑2 and build the vocabulary

In [ ]:
import os, json, urllib.request, pathlib, zipfile, torch, torch.nn as nn, torch.optim as optim, math, random
from torch.utils.data import DataLoader, Dataset

# Create a folder for data
data_dir = pathlib.Path('data')
data_dir.mkdir(exist_ok=True)

# Download WikiText‑2 (≈ 5 MB)
wikitext_url = 'https://s3.amazonaws.com/research.metamind.io/wikitext/wikitext-2-v1.zip'
zip_path = data_dir / 'wikitext-2.zip'
if not zip_path.exists():
    print('Downloading WikiText‑2 …')
    urllib.request.urlretrieve(wikitext_url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(data_dir)
    print('Done.')
else:
    print('WikiText‑2 already downloaded.')

# Load the training split
train_path = data_dir / 'wikitext-2' / 'wiki.train.tokens'
with open(train_path, 'r', encoding='utf‑8') as f:
    raw_text = f.read().replace('\n', ' <eos> ')

# Tokenise and build a simple word‑level vocabulary
tokens = raw_text.split()
vocab = {tok: i+4 for i, tok in enumerate(sorted(set(tokens)))}
vocab['<pad>'] = 0
vocab['<sos>'] = 1
vocab['<eos>'] = 2
vocab['<unk>'] = 3
idx2word = {i: w for w, i in vocab.items()}

# Save the vocabulary for later use
vocab_path = data_dir / 'vocab.json'
with open(vocab_path, 'w') as f:
    json.dump(vocab, f)
print('Vocabulary size:', len(vocab))

## 2. Define the LSTM‑Attention model

In [ ]:
class LSTMAttnLM(nn.Module):
    def __init__(self, vocab_sz, emb_dim=128, hidden=256):
        super().__init__()
        self.emb = nn.Embedding(vocab_sz, emb_dim, padding_idx=vocab['<pad>'])
        self.encoder = nn.LSTM(emb_dim, hidden, batch_first=True)
        # additive (Bahdanau) attention parameters
        self.W1 = nn.Linear(hidden, hidden, bias=False)
        self.W2 = nn.Linear(hidden, hidden, bias=False)
        self.v  = nn.Linear(hidden, 1, bias=False)
        self.decoder = nn.LSTMCell(emb_dim + hidden, hidden)
        self.out = nn.Linear(hidden, vocab_sz)

    def forward(self, src_ids):
        src_emb = self.emb(src_ids)
        enc_out, (h_n, _) = self.encoder(src_emb)  # (B,T,H)
        # Use the final encoder hidden state as initial decoder state
        dec_h = h_n.squeeze(0)
        dec_c = torch.zeros_like(dec_h)
        # Compute attention for the first decoding step
        score = self.v(torch.tanh(self.W1(enc_out) + self.W2(dec_h).unsqueeze(1)))
        attn = torch.softmax(score.squeeze(-1), dim=1)
        context = torch.sum(attn.unsqueeze(-1) * enc_out, dim=1)
        # Input token for the first step is <sos>
        sos = torch.full((src_ids.size(0),), vocab['<sos>'], dtype=torch.long, device=src_ids.device)
        sos_emb = self.emb(sos)
        dec_input = torch.cat([sos_emb, context], dim=1)
        dec_h, dec_c = self.decoder(dec_input, (dec_h, dec_c))
        logits = self.out(dec_h)
        return logits, attn

## 3. Prepare a tiny dataset for training (single‑step prediction)

In [ ]:
class SeqDataset(Dataset):
    def __init__(self, token_list, seq_len=30):
        self.seq_len = seq_len
        self.data = [vocab.get(t, vocab['<unk>']) for t in token_list]

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        src = self.data[idx:idx+self.seq_len]
        tgt = self.data[idx+1:idx+self.seq_len+1]
        return torch.tensor(src), torch.tensor(tgt)

train_dataset = SeqDataset(tokens, seq_len=30)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

## 4. Train the model (a few epochs are enough for demonstration)

In [ ]:
model = LSTMAttnLM(vocab_sz=len(vocab))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 4
for epoch in range(1, epochs+1):
    model.train()
    total_loss = 0.0
    for src, tgt in train_loader:
        optimizer.zero_grad()
        logits, _ = model(src)
        # Predict only the first token after the source sequence
        loss = criterion(logits, tgt[:,0])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch}/{epochs} – loss: {total_loss/len(train_loader):.4f}')

# Save the trained checkpoint for later use
ckpt_path = data_dir / 'lstm_attn_wikitext.pt'
torch.save({'model': model.state_dict()}, ckpt_path)
print('Checkpoint saved to', ckpt_path)

## 5. Prompt engineering – create a prompt and convert it to IDs

In [ ]:
templates = [
    'Complete the sentence: {}',
    'What comes next? {}',
    'Continue: {}'
]

def build_prompt(text, tmpl_id=0):
    prompt = templates[tmpl_id].format(text)
    tokens = prompt.lower().split()
    ids = [vocab.get(tok, vocab['<unk>']) for tok in tokens]
    return torch.tensor([ids], dtype=torch.long)

## 6. Inference – predict the next word and visualise attention

In [ ]:
def predict_next_word(prompt_ids):
    model.eval()
    with torch.no_grad():
        logits, attn = model(prompt_ids)
        prob = torch.softmax(logits, dim=-1)
        top_idx = prob.argmax(dim=-1).item()
        return idx2word[top_idx], attn.squeeze(0).cpu().numpy()

# Example usage
user_text = 'the quick brown fox'
prompt_ids = build_prompt(user_text, tmpl_id=0)
next_word, attn_weights = predict_next_word(prompt_ids)
print('Prompt :', templates[0].format(user_text))
print('Predicted next word →', next_word)

In [ ]:
import matplotlib.pyplot as plt

def show_attention(prompt, attn_weights):
    words = prompt.lower().split()
    plt.figure(figsize=(6,1))
    plt.imshow(attn_weights[:len(words)][None, :], cmap='viridis')
    plt.xticks(range(len(words)), words, rotation=45, ha='right')
    plt.yticks([])
    plt.title('Attention over prompt (first decoding step)')
    plt.show()

show_attention(templates[0].format(user_text), attn_weights)

---
### What you have now
- A tiny LSTM‑Attention language model trained on WikiText‑2.
- Several prompt templates you can modify.
- A function that predicts the first word after a prompt (zero‑shot).
- A visualisation of the attention distribution.

Feel free to change the prompt, try different templates, or extend the decoder loop to generate full sentences.